# 2. Data Preparation and Experiments

This notebook contains data preparation for training recommendation models, experiments for model selection, training the best models on all data (without train/test split), and an example of predicting movie recommendations using the trained models.

## 2.1. Importing Libraries and Loading Paths

In [1]:
import json
import re
from functools import reduce
from itertools import product
from pathlib import Path
from typing import Any
import joblib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sparse
from implicit.cpu.als import AlternatingLeastSquares
from implicit.nearest_neighbours import bm25_weight
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.pipeline import Pipeline
from tqdm import tqdm

from mrh.utils import load_json, save_json
from mrh.data import preprocess_tags, save_data
from mrh.models import save_model

seed = 42

PROJECT_ROOT = Path.cwd().parent
paths_path = PROJECT_ROOT / 'configs/paths.json'

paths = load_json(paths_path)

In [2]:
data_raw_dir = PROJECT_ROOT / paths['data_raw_dir']

ml_32m = data_raw_dir / paths['ml_32m']
ratings_path = ml_32m / paths['ratings_path']
movies_path = ml_32m / paths['movies_path']
tags_path = ml_32m / paths['tags_path']

ratings = pd.read_csv(ratings_path)
movies = pd.read_csv(movies_path)
tags = pd.read_csv(tags_path)

## 2.2. Feature Preprocessing for Movies (genres, year, tags)

In [3]:
tags[tags.isna().any(axis=1)]

,userId,movieId,tag,timestamp
185377,27046,33826,NaN,1221450908
1394089,89369,281500,NaN,1670942104
1914668,153443,123,NaN,1199450867
1914669,153443,346,NaN,1199451946
1914673,153443,1184,NaN,1199452261
1914680,153443,1785,NaN,1199452006
1914681,153443,2194,NaN,1199450677
1914683,153443,2691,NaN,1199451002
1914691,153443,4103,NaN,1199451920
1914693,153443,4473,NaN,1199451040


In [4]:
tags[tags.isnull()] = ''
tags[tags.isna().any(axis=1)]

,userId,movieId,tag,timestamp


In [5]:
tags[tags['tag'] == '']

,userId,movieId,tag,timestamp
185377,27046,33826,,1221450908
1394089,89369,281500,,1670942104
1914668,153443,123,,1199450867
1914669,153443,346,,1199451946
1914673,153443,1184,,1199452261
1914680,153443,1785,,1199452006
1914681,153443,2194,,1199450677
1914683,153443,2691,,1199451002
1914691,153443,4103,,1199451920
1914693,153443,4473,,1199451040


In [6]:
feature_prep = movies.copy().set_index('movieId')
feature_prep['year'] = feature_prep['title'].str.extract(r'\((\d{4})\)')
feature_prep['year'] = pd.to_numeric(feature_prep['year'], errors='coerce').astype('Int64')
print(f'Number of movies without a year: {np.sum(feature_prep["year"].isna())}')
decs = np.ceil((feature_prep['year'].max() - feature_prep['year'].min()) / 10).astype(int)
print(f'Number of decades: {decs}')
feature_prep['year'].describe()

Количество фильмов без года: 615
Количество десятилетий: 15


count        86970.0
mean     1995.354226
std        26.011089
min           1874.0
25%           1981.0
50%           2006.0
75%           2015.0
max           2023.0
Name: year, dtype: Float64

In [7]:
feature_prep['genres_fixed'] = feature_prep['genres'].str.replace('|', ' ').str.lower()
feature_prep['genres_fixed']

movieId
1         adventure animation children comedy fantasy
2                          adventure children fantasy
3                                      comedy romance
4                                comedy drama romance
5                                              comedy
                             ...                     
292731                                          drama
292737                                   comedy drama
292753                                          drama
292755                                          drama
292757                   action adventure documentary
Name: genres_fixed, Length: 87585, dtype: object

In [8]:
feature_prep['decade'] = (feature_prep['year'] // 10).astype(str).str.replace('.', '') + '0s'
feature_prep['decade'] = feature_prep['decade'].fillna('')
movie_tags_groups = tags.groupby('movieId')
tags_dict = {movieId: ' '.join(movie_tags_groups.get_group(movieId)['tag'].to_list()) for movieId in movie_tags_groups.groups.keys()}
feature_prep['tags'] = feature_prep.index.map(tags_dict)
feature_prep['tags']

movieId
1         children Disney animation children Disney Disn...
2         Robin Williams fantasy Robin Williams time tra...
3         comedinha de velhinhos engraÃƒÂ§ada comedinha ...
4         characters slurs based on novel or book chick ...
5         Fantasy pregnancy remake family Steve Martin s...
                                ...                        
292731                                                  NaN
292737                                                  NaN
292753                                                  NaN
292755                                                  NaN
292757                                                  NaN
Name: tags, Length: 87585, dtype: object

In [9]:
feature_prep['tags'].isna().sum()

np.int64(36262)

In [10]:
movies['movieId'].unique().shape[0] - tags['movieId'].unique().shape[0]

36262

In [11]:
feature_prep['tags_clean'] = feature_prep['tags'].apply(preprocess_tags)
feature_prep

,title,genres,year,genres_fixed,decade,tags,tags_clean
movieId,,,,,,,
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,adventure animation children comedy fantasy,1990s,children Disney animation children Disney Disn...,children disney animation children disney disn...
2,Jumanji (1995),Adventure|Children|Fantasy,1995,adventure children fantasy,1990s,Robin Williams fantasy Robin Williams time tra...,robin williams fantasy robin williams time tra...
3,Grumpier Old Men (1995),Comedy|Romance,1995,comedy romance,1990s,comedinha de velhinhos engraÃƒÂ§ada comedinha ...,comedinha de velhinhos engraada comedinha de v...
4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,comedy drama romance,1990s,characters slurs based on novel or book chick ...,characters slurs based on novel or book chick ...
5,Father of the Bride Part II (1995),Comedy,1995,comedy,1990s,Fantasy pregnancy remake family Steve Martin s...,fantasy pregnancy remake family steve martin s...
...,...,...,...,...,...,...,...
292731,The Monroy Affaire (2022),Drama,2022,drama,2020s,NaN,
292737,Shelter in Solitude (2023),Comedy|Drama,2023,comedy drama,2020s,NaN,
292753,Orca (2023),Drama,2023,drama,2020s,NaN,


In [12]:
feature_prep['genres_decade'] = (feature_prep['genres_fixed'] + ' ' + feature_prep['decade']).str.strip()
feature_prep['genres_decade_tags'] = (feature_prep['genres_decade'] + ' ' + feature_prep['tags_clean']).str.strip()
feature_prep

,title,genres,year,genres_fixed,decade,tags,tags_clean,genres_decade,genres_decade_tags
movieId,,,,,,,,,
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,adventure animation children comedy fantasy,1990s,children Disney animation children Disney Disn...,children disney animation children disney disn...,adventure animation children comedy fantasy 1990s,adventure animation children comedy fantasy 19...
2,Jumanji (1995),Adventure|Children|Fantasy,1995,adventure children fantasy,1990s,Robin Williams fantasy Robin Williams time tra...,robin williams fantasy robin williams time tra...,adventure children fantasy 1990s,adventure children fantasy 1990s robin william...
3,Grumpier Old Men (1995),Comedy|Romance,1995,comedy romance,1990s,comedinha de velhinhos engraÃƒÂ§ada comedinha ...,comedinha de velhinhos engraada comedinha de v...,comedy romance 1990s,comedy romance 1990s comedinha de velhinhos en...
4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,comedy drama romance,1990s,characters slurs based on novel or book chick ...,characters slurs based on novel or book chick ...,comedy drama romance 1990s,comedy drama romance 1990s characters slurs ba...
5,Father of the Bride Part II (1995),Comedy,1995,comedy,1990s,Fantasy pregnancy remake family Steve Martin s...,fantasy pregnancy remake family steve martin s...,comedy 1990s,comedy 1990s fantasy pregnancy remake family s...
...,...,...,...,...,...,...,...,...,...
292731,The Monroy Affaire (2022),Drama,2022,drama,2020s,NaN,,drama 2020s,drama 2020s
292737,Shelter in Solitude (2023),Comedy|Drama,2023,comedy drama,2020s,NaN,,comedy drama 2020s,comedy drama 2020s
292753,Orca (2023),Drama,2023,drama,2020s,NaN,,drama 2020s,drama 2020s


In [13]:
data_processed_dir = PROJECT_ROOT / paths['data_processed_dir']
movies_processed_path = data_processed_dir / paths['movies_processed']
save_data(movies_processed_path, feature_prep)

The above preprocessing of movie features includes:
- extracting the year and converting it to a decade string feature;
- converting genres to lowercase and splitting by spaces;
- aggregating corresponding tags into strings for movies, converting to lowercase and keeping only letters, spaces, and digits;
- creating a feature for genres and decade;
- creating a feature for genres, decade, and tags.

Missing values are also handled.

These text features will be used to create numerical features for the content-based recommendation model.

## 2.3. Experiments

### 2.3.1. Functions Required for Experiments

#### 2.3.1.1. Functions for Data Splitting and Displaying Sample Information

In [1]:
def train_test_time_split(ratings, train_size=0.8, model_type='cb'):
    """
    Split data into training and test sets based on time.
    
    For each user, the first train_size% of interactions (by time) go to train,
    the rest go to test. Users with fewer than 5 interactions are filtered out.
    
    Parameters
    ----------
    ratings : pandas.DataFrame
        DataFrame with ratings, must contain columns 'userId', 'movieId', 'timestamp'
    train_size : float, default=0.8
        Proportion of data for the training set (from 0 to 1)
    model_type : str, default='cb'
        Model type:
        - 'cb' (content-based) - returns mappings for train and test movies
        - 'cf' (collaborative filtering) - returns mappings for train movies and users
    
    Returns
    -------
    tuple
        For model_type='cb': (train_df, test_df, train_movieId_to_idx, test_movieId_to_idx)
        For model_type='cf': (train_df, test_df, train_movieId_to_idx, userId_to_idx)
    
    Raises
    ------
    ValueError
        If model_type is not 'cb' or 'cf'
        If train_size is not in range (0, 1)
        If ratings is missing required columns
    TypeError
        If ratings is not a pandas DataFrame
    RuntimeError
        If the test or training set is empty
    """
    if not isinstance(ratings, pd.DataFrame):
        raise TypeError(f"ratings must be pandas.DataFrame, got {type(ratings)}")
    
    required_columns = {'userId', 'movieId', 'timestamp'}
    missing_columns = required_columns - set(ratings.columns)
    if missing_columns:
        raise ValueError(f"Ratings missing required columns: {missing_columns}")
    
    if not 0 < train_size < 1:
        raise ValueError(f"train_size must be in range (0, 1), got {train_size}")

    if model_type not in ['cb', 'cf']:
        raise ValueError(f"model_type must be 'cb' or 'cf', got '{model_type}'")
    
    ratings = ratings.sort_values(['userId', 'timestamp']).reset_index(drop=True)
    
    user_counts = ratings.groupby('userId').size()
    
    valid_users = user_counts[user_counts >= 5].index
    if len(valid_users) == 0:
        raise ValueError("No users with 5 or more ratings after filtering")
    
    ratings_filtered = ratings[ratings['userId'].isin(valid_users)].copy()

    ratings_filtered['user_idx'] = ratings_filtered.groupby('userId').cumcount()
    
    split_sizes = (user_counts[valid_users] * train_size).astype(int)
    
    split_sizes = split_sizes.clip(lower=1)
    
    train_mask = ratings_filtered['user_idx'] < ratings_filtered['userId'].map(split_sizes)
    
    train_df = ratings_filtered[train_mask].drop(columns=['user_idx']).reset_index(drop=True)
    test_df = ratings_filtered[~train_mask].drop(columns=['user_idx']).reset_index(drop=True)
    
    if len(train_df) == 0:
        raise RuntimeError("Training set is empty.")
    if len(test_df) == 0:
        raise RuntimeError("Test set is empty.")

    unique_movies = train_df['movieId'].unique()
    
    train_movieId_to_idx = {int(mid): i for i, mid in enumerate(unique_movies)}
    
    if model_type == 'cf':
        test_df = test_df[test_df['movieId'].isin(unique_movies)]
        
        unique_users = train_df['userId'].unique()
        userId_to_idx = {int(uid): i for i, uid in enumerate(unique_users)}
        
        return train_df, test_df, train_movieId_to_idx, userId_to_idx
        
    elif model_type == 'cb':
        test_unique_movies = test_df['movieId'].unique()
        test_movieId_to_idx = {int(mid): i for i, mid in enumerate(test_unique_movies)}
        
        return train_df, test_df, train_movieId_to_idx, test_movieId_to_idx


def inverse_dict(dictionary):
    """
    Invert a dictionary: swap keys and values.
    
    Parameters
    ----------
    dictionary : dict
        Dictionary to invert. Assumes values are unique.
    
    Returns
    -------
    dict
        Inverted dictionary where the original dictionary's keys become values,
        and values become keys.
    
    Raises
    ------
    TypeError
        If the input parameter is not a dictionary
    """
    if not isinstance(dictionary, dict):
        raise TypeError(f"Expected dictionary, got {type(dictionary).__name__}")
    
    return {v: k for k, v in dictionary.items()}


def print_split_cb_info(train_ratings, test_ratings, train_movieId_to_idx, test_movieId_to_idx, title=''):
    """
    Print information about the data split for the content-based (CB) model.
    
    This function displays statistics about the training and test set sizes,
    as well as information about movies in both datasets.
    
    Parameters
    ----------
    train_ratings : pandas.DataFrame
        Training set with ratings (must contain 'movieId' column)
    test_ratings : pandas.DataFrame
        Test set with ratings (must contain 'movieId' column)
    train_movieId_to_idx : dict
        Dictionary mapping movie ID to index for the training set
    test_movieId_to_idx : dict
        Dictionary mapping movie ID to index for the test set
    title : str, default=''
        Additional title or prefix for the displayed information
    
    Returns
    -------
    None
    
    Raises
    ------
    TypeError
        If any of the input parameters has an incorrect type
    """
    if not hasattr(train_ratings, 'shape'):
        raise TypeError(f"train_ratings must be a DataFrame or similar object with shape attribute, got {type(train_ratings)}")
    
    if not hasattr(test_ratings, 'shape'):
        raise TypeError(f"test_ratings must be a DataFrame or similar object with shape attribute, got {type(test_ratings)}")
    
    if not isinstance(train_movieId_to_idx, dict):
        raise TypeError(f"train_movieId_to_idx must be a dictionary, got {type(train_movieId_to_idx)}")
    
    if not isinstance(test_movieId_to_idx, dict):
        raise TypeError(f"test_movieId_to_idx must be a dictionary, got {type(test_movieId_to_idx)}")
    
    if not isinstance(title, str):
        raise TypeError(f"title must be a string, got {type(title)}")
    
    print(f'Total {title}ratings: {train_ratings.shape[0] + test_ratings.shape[0]:_}')
    print(f'Training set size: {train_ratings.shape[0]:_}')
    print(f'Test set size: {test_ratings.shape[0]:_}')
    print(f'Number of movies in training set: {len(train_movieId_to_idx.keys())}')
    print(f'Number of movies in test set: {len(test_movieId_to_idx.keys())}')
    
    num_intersect_m = len(set(train_movieId_to_idx.keys()) & set(test_movieId_to_idx.keys()))
    print(f'Number of movies overlapping between training and test sets: {num_intersect_m:_}')
    
    num_union_m = len(set(train_movieId_to_idx.keys()) | set(test_movieId_to_idx.keys()))
    print(f'Total movies: {num_union_m:_}')


def print_split_cf_info(train_ratings, test_ratings, movieId_to_idx, userId_to_idx, title=''):
    """
    Print information about the data split for the collaborative (CF) model.
    
    Parameters
    ----------
    train_ratings : pandas.DataFrame
        Training set with ratings
    test_ratings : pandas.DataFrame
        Test set with ratings
    movieId_to_idx : dict
        Dictionary mapping movie ID to index
    userId_to_idx : dict
        Dictionary mapping user ID to index
    title : str, default=''
        Additional title or prefix for the displayed information
    
    Returns
    -------
    None
        This function only prints information to the console, nothing is returned
    
    Raises
    ------
    TypeError
        If the input parameters have incorrect types
    """
    # Type checking
    if not hasattr(train_ratings, 'shape'):
        raise TypeError(f"train_ratings must have a shape attribute, got {type(train_ratings)}")
    
    if not hasattr(test_ratings, 'shape'):
        raise TypeError(f"test_ratings must have a shape attribute, got {type(test_ratings)}")
    
    if not isinstance(movieId_to_idx, dict):
        raise TypeError(f"movieId_to_idx must be a dictionary, got {type(movieId_to_idx)}")
    
    if not isinstance(userId_to_idx, dict):
        raise TypeError(f"userId_to_idx must be a dictionary, got {type(userId_to_idx)}")
    
    if not isinstance(title, str):
        raise TypeError(f"title must be a string, got {type(title)}")
    
    print(f'Total {title}ratings: {train_ratings.shape[0] + test_ratings.shape[0]:_}')
    print(f'Training set size: {train_ratings.shape[0]:_}')
    print(f'Test set size: {test_ratings.shape[0]:_}')
    print(f'Number of movies: {len(movieId_to_idx.keys())}')
    print(f'Number of users: {len(userId_to_idx.keys())}')

#### 2.3.1.2. Functions for Calculating Ranking Quality Metrics and Helper Functions

In [2]:
def metrics_at_k_random(all_movie_ids, ground_truth_dict, user_ids=None, k=10, seed=42):
    """
    Calculate quality metrics for a random recommendation model (baseline).
    
    For each user, randomly selects k movies from the general list,
    then calculates precision, recall, hit_rate and NDCG based on ground truth data.
    The random model is used as a baseline for comparison with other approaches.
    
    Parameters
    ----------
    all_movie_ids : list or numpy.ndarray
        List of all available movie IDs from which random selection will occur
    ground_truth_dict : dict
        Dictionary with ground truth data for each user.
        Each element must contain keys:
        - 'movieIds': list of movie IDs rated by the user
        - 'ratings': dictionary {movieId: rating}
    user_ids : list, optional, default=None
        List of user IDs to evaluate. If None, all users from ground_truth_dict are used
    k : int, default=10
        Number of recommendations per user
    seed : int, default=42
        Seed parameter for reproducibility
    
    Returns
    -------
    tuple
        (precisions, recalls, hit_rates, ndcgs, k)
        - precisions : dict {user_id: precision@k}
        - recalls : dict {user_id: recall@k}
        - hit_rates : dict {user_id: hit_rate@k}
        - ndcgs : dict {user_id: ndcg@k}
        - k : int (returned for convenience)
    
    Raises
    ------
    TypeError
        If input parameters have incorrect type
    ValueError
        If k <= 0, if all_movie_ids is empty, or if there are no users to evaluate
    KeyError
        If ground_truth_dict is missing required keys
    RuntimeError
        If the number of evaluated predictions is 0
    """
    if not isinstance(all_movie_ids, (list, np.ndarray)):
        raise TypeError(f"all_movie_ids must be a list or numpy array, got {type(all_movie_ids)}")
    
    if len(all_movie_ids) == 0:
        raise ValueError("all_movie_ids cannot be empty")
    
    if not isinstance(ground_truth_dict, dict):
        raise TypeError(f"ground_truth_dict must be a dictionary, got {type(ground_truth_dict)}")
    
    if len(ground_truth_dict) == 0:
        raise ValueError("ground_truth_dict cannot be empty")
    
    if user_ids is not None and not isinstance(user_ids, (list, np.ndarray)):
        raise TypeError(f"user_ids must be a list, numpy array or None, got {type(user_ids)}")
    
    if not isinstance(k, int) or k <= 0:
        raise ValueError(f"k must be a positive integer, got {k}")
    
    if not isinstance(seed, int):
        raise TypeError(f"seed must be an integer, got {type(seed)}")
    
    try:
        rng = np.random.default_rng(seed)
    except Exception as e:
        raise RuntimeError(f"Error initializing random number generator with seed={seed}: {e}")

    if user_ids is None:
        user_ids = list(ground_truth_dict.keys())
    else:
        missing_users = set(user_ids) - set(ground_truth_dict.keys())
        if missing_users:
            import warnings
            warnings.warn(f"The following users are missing from ground_truth_dict and will be skipped: {missing_users}", UserWarning)
    
    for uid in user_ids:
        if uid in ground_truth_dict:
            if not isinstance(ground_truth_dict[uid], dict):
                raise TypeError(f"Data for user {uid} must be a dictionary, got {type(ground_truth_dict[uid])}")
            if 'movieIds' not in ground_truth_dict[uid]:
                raise KeyError(f"ground_truth_dict for user {uid} missing key 'movieIds'")
            if 'ratings' not in ground_truth_dict[uid]:
                raise KeyError(f"ground_truth_dict for user {uid} missing key 'ratings'")
    
    actual_k = min(k, len(all_movie_ids))
    if actual_k < k:
        import warnings
        warnings.warn(f"k={k} exceeds the number of available movies ({len(all_movie_ids)}). "
                      f"Will use actual_k={actual_k}", UserWarning)
    valid_users = []
    skipped_users = []
    
    for uid in user_ids:
        if uid not in ground_truth_dict:
            skipped_users.append(uid)
            continue
        
        truth_data = ground_truth_dict[uid]
        movie_ids = truth_data.get('movieIds', [])
        
        if not isinstance(movie_ids, list) and not isinstance(movie_ids, set):
            import warnings
            warnings.warn(f"User {uid} field 'movieIds' is not a list, user skipped", UserWarning)
            skipped_users.append(uid)
            continue
        
        if len(movie_ids) == 0:
            skipped_users.append(uid)
            continue
        
        valid_users.append(uid)
    
    if len(valid_users) == 0:
        raise ValueError("No valid users for evaluation (all have empty 'movieIds' list)")
    
    if skipped_users and len(skipped_users) < 10:
        import warnings
        warnings.warn(f"Skipped users: {skipped_users}", UserWarning)
    elif skipped_users:
        print(f"Info: skipped {len(skipped_users)} users (no data or empty movie list)")
    
    precisions = {}
    recalls = {}
    hit_rates = {}
    ndcgs = {}

    log_weights = np.array([1.0 / np.log2(i + 2) for i in range(k)])
    
    for user_id in tqdm(valid_users, desc="Random Baseline"):
        try:
            if actual_k == len(all_movie_ids):
                recommendations = all_movie_ids.copy()
                rng.shuffle(recommendations)
            else:
                recommendations = rng.choice(all_movie_ids, size=actual_k, replace=False).tolist()
            
            truth_data = ground_truth_dict[user_id]
            true_set = truth_data['movieIds']
            num_true = len(true_set)
            ratings = truth_data['ratings']
            
            rec_set = set(recommendations)
            hits = len(rec_set & true_set)

            precisions[user_id] = hits / k
            recalls[user_id] = hits / num_true if num_true > 0 else 0
            hit_rates[user_id] = 1 if hits > 0 else 0

            try:
                dcg = sum(ratings.get(mid, 0) * log_weights[j] 
                         for j, mid in enumerate(recommendations) if j < len(log_weights))
                
                ideal_ratings = sorted(ratings.values(), reverse=True)[:k]
                idcg = sum(r * log_weights[j] for j, r in enumerate(ideal_ratings) if j < len(log_weights))
                ndcgs[user_id] = float(dcg / idcg) if idcg > 0 else 0
            except Exception as e:
                import warnings
                warnings.warn(f"Error calculating NDCG for user {user_id}: {e}", UserWarning)
                ndcgs[user_id] = 0
            
        except Exception as e:
            import warnings
            warnings.warn(f"Error processing user {user_id}: {e}, user skipped", UserWarning)
            continue

    if len(precisions) == 0:
        raise RuntimeError("Failed to process any users")
    
    return precisions, recalls, hit_rates, ndcgs, k


def metrics_at_k_CB(user_profiles, test_features, ground_truth_dict, test_idx_to_movieId, k=10, batch_size=1000, verbose=True):
    """
    Calculate recommendation quality metrics for content-based (CB) model at k.
    
    For each user, calculates precision, recall, hit_rate and NDCG based on
    cosine similarity between user profile and test item features.
    
    Parameters
    ----------
    user_profiles : dict
        Dictionary where key is user ID, value is user vector profile (numpy array)
    test_features : numpy.ndarray or scipy.sparse matrix
        Feature matrix of test items (size: n_items x n_features)
    ground_truth_dict : dict
        Dictionary with ground truth data for each user.
        Each element must contain keys 'movieIds' (list of movie IDs)
        and 'ratings' (dictionary {movieId: rating})
    test_idx_to_movieId : dict
        Dictionary mapping test item index to its original ID
    k : int, default=10
        Number of recommendations per user
    batch_size : int, default=1000
        Batch size for cosine similarity calculation
    verbose : bool, default=True
        Whether to show progress bar with tqdm
    
    Returns
    -------
    tuple
        (precisions, recalls, hit_rates, ndcgs, k)
        - precisions : dict {user_id: precision@k}
        - recalls : dict {user_id: recall@k}
        - hit_rates : dict {user_id: hit_rate@k}
        - ndcgs : dict {user_id: ndcg@k}
        - k : int (returned for convenience)
    
    Raises
    ------
    TypeError
        If input parameters have incorrect type
    ValueError
        If k <= 0, batch_size <= 0, or if matrices have incompatible dimensions, or if user_profiles is empty
    KeyError
        If ground_truth_dict is missing required keys
    """
    if not isinstance(user_profiles, dict):
        raise TypeError(f"user_profiles must be a dictionary, got {type(user_profiles)}")
    
    if len(user_profiles) == 0:
        raise ValueError("user_profiles cannot be an empty dictionary")
    
    if not isinstance(test_features, (np.ndarray, sparse.spmatrix, pd.DataFrame)):
        raise TypeError(f"test_features must be numpy array, sparse matrix, or pandas DataFrame, got {type(test_features)}")
    
    if not isinstance(ground_truth_dict, dict):
        raise TypeError(f"ground_truth_dict must be a dictionary, got {type(ground_truth_dict)}")
    
    if not isinstance(test_idx_to_movieId, dict):
        raise TypeError(f"test_idx_to_movieId must be a dictionary, got {type(test_idx_to_movieId)}")
    
    if not isinstance(k, int) or k <= 0:
        raise ValueError(f"k must be a positive integer, got {k}")
    
    if not isinstance(batch_size, int) or batch_size <= 0:
        raise ValueError(f"batch_size must be a positive integer, got {batch_size}")
    
    first_user_vec = next(iter(user_profiles.values()))
    if test_features.shape[1] != len(first_user_vec):
        raise ValueError(
            f"Dimension mismatch: test_features has {test_features.shape[1]} features, "
            f"but user profile has {len(first_user_vec)} features"
        )
    
    for user_id, truth_data in ground_truth_dict.items():
        if not isinstance(truth_data, dict):
            raise TypeError(f"Data for user {user_id} must be a dictionary, got {type(truth_data)}")
        if 'movieIds' not in truth_data:
            raise KeyError(f"ground_truth_dict for user {user_id} missing key 'movieIds'")
        if 'ratings' not in truth_data:
            raise KeyError(f"ground_truth_dict for user {user_id} missing key 'ratings'")
    
    precisions = {}
    recalls = {}
    hit_rates = {}
    ndcgs = {}

    log_weights = np.array([1.0 / np.log2(i + 2) for i in range(k)])
    
    user_items = list(user_profiles.items())
    
    for batch_start in tqdm(range(0, len(user_items), batch_size), disable=not verbose, desc="Batches"):
        batch = user_items[batch_start:batch_start + batch_size]
        
        batch_ids = [uid for uid, _ in batch]
        batch_vectors_list = []
        for _, vec in batch:
            if isinstance(vec, sparse.spmatrix):
                vec = vec.toarray().flatten()
            batch_vectors_list.append(vec)
        
        batch_vectors = np.vstack(batch_vectors_list)
        
        try:
            batch_similarities = cosine_similarity(batch_vectors, test_features)
        except Exception as e:
            raise RuntimeError(f"Error calculating cosine similarity: {e}")
        
        for i, user_id in enumerate(batch_ids):
            similarities = batch_similarities[i]
            
            if len(similarities) >= k:
                top_k_idx = np.argpartition(similarities, -k)[-k:]
                top_k_idx = top_k_idx[np.argsort(-similarities[top_k_idx])]
            else:
                top_k_idx = np.argsort(-similarities)[:len(similarities)]

            recommendations = []
            for idx in top_k_idx:
                if idx < len(test_idx_to_movieId):
                    recommendations.append(test_idx_to_movieId[idx])
                else:
                    import warnings
                    warnings.warn(f"Index {idx} is out of bounds for test_idx_to_movieId", UserWarning)
            
            if user_id not in ground_truth_dict:
                import warnings
                warnings.warn(f"User {user_id} not found in ground_truth_dict", UserWarning)
                precisions[user_id] = 0
                recalls[user_id] = 0
                hit_rates[user_id] = 0
                ndcgs[user_id] = 0
                continue
            
            truth_data = ground_truth_dict[user_id]
            true_set = set(truth_data['movieIds'])
            num_true = len(true_set)
            
            rec_set = set(recommendations)
            hits = len(rec_set & true_set)
            
            precisions[user_id] = hits / k if k > 0 else 0
            recalls[user_id] = hits / num_true if num_true > 0 else 0
            hit_rates[user_id] = 1 if hits > 0 else 0

            ratings = truth_data['ratings']
            dcg = 0.0
            for j, mid in enumerate(recommendations):
                if j < len(log_weights):
                    dcg += ratings.get(mid, 0) * log_weights[j]
            
            ideal_ratings = sorted(ratings.values(), reverse=True)[:k]
            idcg = sum(r * log_weights[j] for j, r in enumerate(ideal_ratings) if j < len(log_weights))
            ndcgs[user_id] = float(dcg / idcg) if idcg > 0 else 0
    
    return precisions, recalls, hit_rates, ndcgs, k


def metrics_at_k_ALS(model, train_matrix, test_matrix, userIds, userId_to_idx, idx_to_userId, movieId_to_idx, idx_to_movieId, k=10, filter_already_liked=True, batch_size=1000):
    """
    Calculate quality metrics for ALS model (Alternating Least Squares).
    
    For each user, gets recommendations from the ALS model and compares
    with ground truth data from the test matrix.
    
    Parameters
    ----------
    model : implicit.als.AlternatingLeastSquares
        Trained ALS model
    train_matrix : scipy.sparse.csr_matrix
        Training interaction matrix (users x movies)
    test_matrix : scipy.sparse.csr_matrix
        Test interaction matrix (users x movies)
    userIds : list
        List of user IDs to evaluate
    userId_to_idx : dict
        Dictionary mapping user ID to index
    idx_to_userId : dict
        Dictionary mapping index to user ID
    movieId_to_idx : dict
        Dictionary mapping movie ID to index
    idx_to_movieId : dict
        Dictionary mapping index to movie ID
    k : int, default=10
        Number of recommendations per user
    filter_already_liked : bool, default=True
        Whether to filter already liked movies from recommendations
    batch_size : int, default=1000
        Batch size for processing users
    
    Returns
    -------
    tuple
        (precisions, recalls, hit_rates, ndcgs, k)
        - precisions : dict {user_id: precision@k}
        - recalls : dict {user_id: recall@k}
        - hit_rates : dict {user_id: hit_rate@k}
        - ndcgs : dict {user_id: ndcg@k}
        - k : int (returned for convenience)
    
    Raises
    ----------
    TypeError
        If input parameters have incorrect type
    ValueError
        If k <= 0 or batch_size <= 0
    RuntimeError
        If no users could be processed
    """
    if not isinstance(k, int) or k <= 0:
        raise ValueError(f"k must be a positive integer, got {k}")
    
    if not isinstance(batch_size, int) or batch_size <= 0:
        raise ValueError(f"batch_size must be a positive integer, got {batch_size}")
    
    if not isinstance(userId_to_idx, dict) or len(userId_to_idx) == 0:
        raise ValueError("userId_to_idx cannot be an empty dictionary")
    
    if not isinstance(idx_to_movieId, dict) or len(idx_to_movieId) == 0:
        raise ValueError("idx_to_movieId cannot be an empty dictionary")
    
    precisions = {}
    recalls = {}
    hit_rates = {}
    ndcgs = {}
    
    log_weights = np.array([1.0 / np.log2(i + 2) for i in range(k)])
    
    user_indices = []
    valid_original_ids = []
    
    for uid in userIds:
        if uid in userId_to_idx:
            user_indices.append(userId_to_idx[uid])
            valid_original_ids.append(uid)
    
    if len(user_indices) == 0:
        raise RuntimeError("No valid users for evaluation")

    valid_users_with_test = []
    valid_original_with_test = []
    
    for user_idx, original_uid in zip(user_indices, valid_original_ids):
        test_user_items = test_matrix[user_idx].nonzero()[1]
        if len(test_user_items) > 0:
            valid_users_with_test.append(user_idx)
            valid_original_with_test.append(original_uid)
    
    if len(valid_users_with_test) == 0:
        raise RuntimeError("No users with data in the test set")
    
    for batch_start in tqdm(range(0, len(valid_users_with_test), batch_size), desc="Batches"):
        batch_end = min(batch_start + batch_size, len(valid_users_with_test))
        batch_user_indices = valid_users_with_test[batch_start:batch_end]
        batch_original_ids = valid_original_with_test[batch_start:batch_end]
        
        for user_idx, original_uid in zip(batch_user_indices, batch_original_ids):
            recommended_movies = []
            
            try:
                train_user_items = train_matrix[user_idx].nonzero()[1]
                
                if len(train_user_items) == 0:
                    continue
                
                recs = model.recommend(
                    user_idx,
                    train_matrix[user_idx],
                    N=k,
                    filter_already_liked_items=filter_already_liked
                )
                
                if recs is None:
                    continue
                
                if len(recs) == 2:
                    movie_ids_array, scores_array = recs
                    for movie_idx in movie_ids_array:
                        if movie_idx in idx_to_movieId:
                            recommended_movies.append(idx_to_movieId[movie_idx])
                elif len(recs) > 0 and isinstance(recs[0], (tuple, list)):
                    for item in recs:
                        if isinstance(item, (tuple, list)) and len(item) >= 2:
                            movie_idx = item[0]
                            if movie_idx in idx_to_movieId:
                                recommended_movies.append(idx_to_movieId[movie_idx])
                            
            except Exception as e:
                continue
            
            test_user_movies = test_matrix[user_idx].nonzero()[1]
            train_user_movies_set = set(train_matrix[user_idx].nonzero()[1])
            
            true_positives_ids = []
            for movie_idx in test_user_movies:
                if movie_idx not in train_user_movies_set:
                    if movie_idx in idx_to_movieId:
                        true_positives_ids.append(idx_to_movieId[movie_idx])
            
            true_set = set(true_positives_ids)
            num_true = len(true_set)
            
            if num_true == 0:
                continue
            
            if len(recommended_movies) == 0:
                precisions[original_uid] = 0.0
                recalls[original_uid] = 0.0
                hit_rates[original_uid] = 0.0
                ndcgs[original_uid] = 0.0
                continue
            
            rec_set = set(recommended_movies[:k])
            hits = len(rec_set & true_set)
            
            precisions[original_uid] = hits / k
            recalls[original_uid] = hits / num_true if num_true > 0 else 0
            hit_rates[original_uid] = 1 if hits > 0 else 0

            dcg = 0.0
            for j, movieId in enumerate(recommended_movies[:k]):
                if movieId in true_set:
                    dcg += 1.0 * log_weights[j]
            
            ideal_dcg = sum(log_weights[j] for j in range(min(num_true, k)))
            ndcgs[original_uid] = float(dcg / ideal_dcg) if ideal_dcg > 0 else 0
    
    if len(precisions) == 0:
        raise RuntimeError("Failed to process any users")
    
    return precisions, recalls, hit_rates, ndcgs, k


def agg_metrics(precisions, recalls, hit_rates, ndcgs, k):
    """
    Aggregate metrics across all users and return average values.
    
    Parameters
    ----------
    precisions : dict
        Dictionary with precision for each user {user_id: precision@k}
    recalls : dict
        Dictionary with recall for each user {user_id: recall@k}
    hit_rates : dict
        Dictionary with hit_rate for each user {user_id: hit_rate@k}
    ndcgs : dict
        Dictionary with ndcg for each user {user_id: ndcg@k}
    k : int
        Number of recommendations (used for naming metrics)
    
    Returns
    -------
    dict
        Dictionary with aggregated metrics:
        - f'precision@{k}': average precision
        - f'recall@{k}': average recall
        - f'hit_rate@{k}': average hit rate
        - f'ndcg@{k}': average NDCG
    
    Raises
    ------
    TypeError
        If input parameters have incorrect type
    ValueError
        If dictionaries are empty or have different numbers of users
    """

    if not isinstance(precisions, dict):
        raise TypeError(f"precisions must be a dictionary, got {type(precisions)}")
    if not isinstance(recalls, dict):
        raise TypeError(f"recalls must be a dictionary, got {type(recalls)}")
    if not isinstance(hit_rates, dict):
        raise TypeError(f"hit_rates must be a dictionary, got {type(hit_rates)}")
    if not isinstance(ndcgs, dict):
        raise TypeError(f"ndcgs must be a dictionary, got {type(ndcgs)}")
    if not isinstance(k, int):
        raise TypeError(f"k must be an integer, got {type(k)}")
    
    if len(precisions) == 0:
        raise ValueError("precisions dictionary cannot be empty")
    
    num_users = len(precisions)
    if len(recalls) != num_users:
        raise ValueError(f"Number of users in recalls ({len(recalls)}) does not match precisions ({num_users})")
    if len(hit_rates) != num_users:
        raise ValueError(f"Number of users in hit_rates ({len(hit_rates)}) does not match precisions ({num_users})")
    if len(ndcgs) != num_users:
        raise ValueError(f"Number of users in ndcgs ({len(ndcgs)}) does not match precisions ({num_users})")
    
    final_metrics = {}
    final_metrics[f'precision@{k}'] = sum(precisions.values()) / num_users
    final_metrics[f'recall@{k}'] = sum(recalls.values()) / num_users
    final_metrics[f'hit_rate@{k}'] = sum(hit_rates.values()) / num_users
    final_metrics[f'ndcg@{k}'] = sum(ndcgs.values()) / num_users
    
    return final_metrics


def make_user_profiles(train_ratings, train_features, train_movieId_to_idx, num_movies, sort_rating=True, sort_timestamp=True):
    """
    Create user profiles based on aggregated features of movies they rated.
    
    For each user, selects top-N movies (by rating and/or time),
    then forms the user profile as the arithmetic mean of the
    feature vectors of the selected movies.
    
    Parameters
    ----------
    train_ratings : pandas.DataFrame
        DataFrame with user ratings. Must contain columns:
        'userId', 'movieId', 'rating', 'timestamp'
    train_features : numpy.ndarray, pandas.DataFrame or scipy.sparse.spmatrix
        Movie feature matrix of size (n_movies x n_features)
    train_movieId_to_idx : dict
        Dictionary mapping movie ID to index in train_features matrix
    num_movies : int
        Number of top movies per user used to build the profile
    sort_rating : bool, default=True
        Whether to sort movies by rating (descending)
    sort_timestamp : bool, default=True
        Whether to sort movies by time (descending - newer first)
    
    Returns
    -------
    dict
        Dictionary of user profiles {user_id: numpy.ndarray of user profile}
    
    Raises
    ------
    TypeError
        If input parameters have incorrect type
    ValueError
        If num_movies <= 0, or if train_ratings is missing required columns,
        or if there are no users to process
    KeyError
        If movieId from train_ratings is missing from train_movieId_to_idx
    RuntimeError
        If no profiles could be created
    """

    if not isinstance(train_ratings, pd.DataFrame):
        raise TypeError(f"train_ratings must be pandas.DataFrame, got {type(train_ratings)}")
    
    if len(train_ratings) == 0:
        raise ValueError("train_ratings cannot be empty")

    required_columns = {'userId', 'movieId', 'rating', 'timestamp'}
    missing_columns = required_columns - set(train_ratings.columns)
    if missing_columns:
        raise ValueError(f"train_ratings missing required columns: {missing_columns}")

    if not isinstance(train_features, (np.ndarray, pd.DataFrame, sparse.spmatrix)):
        raise TypeError(
            f"train_features must be numpy.ndarray, pandas.DataFrame or scipy.sparse.spmatrix, "
            f"got {type(train_features)}"
        )
    
    if not isinstance(train_movieId_to_idx, dict):
        raise TypeError(f"train_movieId_to_idx must be a dictionary, got {type(train_movieId_to_idx)}")
    
    if len(train_movieId_to_idx) == 0:
        raise ValueError("train_movieId_to_idx cannot be empty")
    
    if not isinstance(num_movies, int) or num_movies <= 0:
        raise ValueError(f"num_movies must be a positive integer, got {num_movies}")
    
    if not isinstance(sort_rating, bool):
        raise TypeError(f"sort_rating must be a boolean, got {type(sort_rating)}")
    
    if not isinstance(sort_timestamp, bool):
        raise TypeError(f"sort_timestamp must be a boolean, got {type(sort_timestamp)}")
    
    try:
        if sort_rating and sort_timestamp:
            train_ratings_sorted = train_ratings.sort_values(
                ['userId', 'rating', 'timestamp'], 
                ascending=[True, False, False]
            )
        elif sort_rating and not sort_timestamp:
            train_ratings_sorted = train_ratings.sort_values(
                ['userId', 'rating'], 
                ascending=[True, False]
            )
        elif not sort_rating and sort_timestamp:
            train_ratings_sorted = train_ratings.sort_values(
                ['userId', 'timestamp'], 
                ascending=[True, False]
            )
        else:
            train_ratings_sorted = train_ratings.copy()
    except Exception as e:
        raise RuntimeError(f"Error sorting train_ratings: {e}")

    try:
        top_movies_per_user = (train_ratings_sorted
                               .groupby('userId')
                               .head(num_movies)
                               .reset_index(drop=True))
    except Exception as e:
        raise RuntimeError(f"Error grouping and selecting top movies: {e}")
    
    if len(top_movies_per_user) == 0:
        raise ValueError("No data remained after top movies filtering")
    
    is_sparse = sparse.isspmatrix(train_features)
    is_dataframe = hasattr(train_features, 'iloc')
    
    user_profiles = {}
    skipped_movies = set()
    processed_users = 0
    
    for userId, group in top_movies_per_user.groupby('userId'):
        try:
            movie_indices = []
            for mid in group['movieId']:
                try:
                    if mid not in train_movieId_to_idx:
                        if mid not in skipped_movies:
                            skipped_movies.add(mid)
                        continue
                    movie_indices.append(train_movieId_to_idx[mid])
                except Exception as e:
                    import warnings
                    warnings.warn(f"Error processing movieId {mid} for user {userId}: {e}", UserWarning)
                    continue
            
            if len(movie_indices) == 0:
                import warnings
                warnings.warn(f"User {userId} has no valid movieIds after filtering, skipped", UserWarning)
                continue
            
            if is_sparse:
                features = train_features[movie_indices]
                if len(movie_indices) == 1:
                    user_profile = features.toarray().flatten()
                else:
                    user_profile = features.mean(axis=0).toarray().flatten()
            elif is_dataframe:
                features = train_features.iloc[movie_indices]
                if len(movie_indices) == 1:
                    user_profile = features.values.flatten()
                else:
                    user_profile = features.mean(axis=0).values.flatten()
            else:
                features = train_features[movie_indices]
                if len(movie_indices) == 1:
                    user_profile = features.flatten()
                else:
                    user_profile = features.mean(axis=0).flatten()
            
            norm = np.linalg.norm(user_profile)
            if norm > 0:
                user_profile = user_profile / norm
            
            user_profiles[userId] = user_profile
            processed_users += 1
            
        except Exception as e:
            import warnings
            warnings.warn(f"Error creating profile for user {userId}: {e}", UserWarning)
            continue
    
    if skipped_movies:
        import warnings
        warnings.warn(
            f"The following movieIds are missing from train_movieId_to_idx and were skipped: "
            f"{list(skipped_movies)[:10]}{'...' if len(skipped_movies) > 10 else ''}", 
            UserWarning
        )
    
    if processed_users == 0:
        raise RuntimeError("Failed to create profiles for any user")
    
    total_users = len(top_movies_per_user['userId'].unique())
    if processed_users < total_users:
        import warnings
        warnings.warn(
            f"Created profiles for {processed_users} out of {total_users} users. "
            f"{total_users - processed_users} users skipped due to errors or lack of valid movies",
            UserWarning
        )
    
    return user_profiles


def get_ground_truth(test_ratings):
    """
    Create a ground truth dictionary for the test set.
    
    For each user, creates a dictionary with a set of movie IDs and a rating dictionary.
    
    Parameters
    ----------
    test_ratings : pandas.DataFrame
        DataFrame with test ratings. Must contain columns 'userId', 'movieId', 'rating'
    
    Returns
    -------
    dict
        Dictionary of the form {user_id: {'movieIds': set, 'ratings': dict}},
        where 'movieIds' is a set of movie IDs, 'ratings' is a dictionary {movieId: rating}
    
    Raises
    ----------
    TypeError
        If test_ratings is not a pandas DataFrame
    ValueError
        If test_ratings is missing required columns or data is empty
    RuntimeError
        If the dictionary could not be created
    """

    if not isinstance(test_ratings, pd.DataFrame):
        raise TypeError(f"test_ratings must be pandas.DataFrame, got {type(test_ratings)}")
    
    if len(test_ratings) == 0:
        raise ValueError("test_ratings cannot be empty")
    
    required_columns = {'userId', 'movieId', 'rating'}
    missing_columns = required_columns - set(test_ratings.columns)
    if missing_columns:
        raise ValueError(f"test_ratings missing required columns: {missing_columns}")
    
    ground_truth_dict = {}
    for userId, group in test_ratings.groupby('userId'):
        ground_truth_dict[userId] = {
            'movieIds': set(group['movieId']), 
            'ratings': dict(zip(group['movieId'], group['rating']))
        }
    
    if len(ground_truth_dict) == 0:
        raise RuntimeError("Failed to create ground_truth_dict: no users")
    
    return ground_truth_dict

#### 2.3.1.3. Functions for Creating Vectorizers and Features for Content-based Recommendation Models

In [3]:
def onehot_genres_features(processed_movies, train_movieIds, test_movieIds):
    """
    Create one-hot encoding of genre features for training and test movies.
    
    Parameters
    ----------
    processed_movies : pandas.DataFrame
        DataFrame with movie information, must contain column 'genres_fixed'
    train_movieIds : list or array-like
        List of movie IDs for the training set
    test_movieIds : list or array-like
        List of movie IDs for the test set
    
    Returns
    -------
    tuple
        (mlb, train_genres_df, test_genres_df)
        - mlb : MultiLabelBinarizer - trained binarizer
        - train_genres_df : pandas.DataFrame - one-hot features for training movies
        - test_genres_df : pandas.DataFrame - one-hot features for test movies
    
    Raises
    ----------
    TypeError
        If input parameters have incorrect type
    ValueError
        If column 'genres_fixed' is missing or data is empty
    RuntimeError
        If features could not be created
    """

    if not isinstance(processed_movies, pd.DataFrame):
        raise TypeError(f"processed_movies must be pandas.DataFrame, got {type(processed_movies)}")
    
    if 'genres_fixed' not in processed_movies.columns:
        raise ValueError("Column 'genres_fixed' is missing from processed_movies")
    
    if len(train_movieIds) == 0:
        raise ValueError("train_movieIds cannot be empty")
    
    if len(test_movieIds) == 0:
        raise ValueError("test_movieIds cannot be empty")
    
    try:
        train_genres_lists = processed_movies.loc[train_movieIds, 'genres_fixed'].str.split(' ').tolist()
        test_genres_lists = processed_movies.loc[test_movieIds, 'genres_fixed'].str.split(' ').tolist()
    except Exception as e:
        raise RuntimeError(f"Error extracting genres: {e}")
    
    mlb = MultiLabelBinarizer()
    
    try:
        train_genres_onehot = mlb.fit_transform(train_genres_lists)
        test_genres_onehot = mlb.transform(test_genres_lists)
    except Exception as e:
        raise RuntimeError(f"Error during one-hot encoding: {e}")
    
    train_genres_df = pd.DataFrame(train_genres_onehot, columns=mlb.classes_)
    test_genres_df = pd.DataFrame(test_genres_onehot, columns=mlb.classes_)
    
    return mlb, train_genres_df, test_genres_df


def tfidf_genres_dec_features(processed_movies, train_movieIds, test_movieIds):
    """
    Create TF-IDF features from the 'genres_decade' column (string: genres + decade).
    
    Parameters
    ----------
    processed_movies : pandas.DataFrame
        DataFrame with movie information, must contain column 'genres_decade'
    train_movieIds : list or array-like
        List of movie IDs for the training set
    test_movieIds : list or array-like
        List of movie IDs for the test set
    
    Returns
    -------
    tuple
        (vectorizer, train_tfidf, test_tfidf)
        - vectorizer : TfidfVectorizer - trained vectorizer
        - train_tfidf : scipy.sparse matrix - TF-IDF matrix for training movies
        - test_tfidf : scipy.sparse matrix - TF-IDF matrix for test movies
    
    Raises
    ----------
    TypeError
        If input parameters have incorrect type
    ValueError
        If column 'genres_decade' is missing or data is empty
    RuntimeError
        If TF-IDF features could not be created
    """
    if not isinstance(processed_movies, pd.DataFrame):
        raise TypeError(f"processed_movies must be pandas.DataFrame, got {type(processed_movies)}")
    
    if 'genres_decade' not in processed_movies.columns:
        raise ValueError("Column 'genres_decade' is missing from processed_movies")
    
    if len(train_movieIds) == 0:
        raise ValueError("train_movieIds cannot be empty")
    
    if len(test_movieIds) == 0:
        raise ValueError("test_movieIds cannot be empty")
    
    try:
        train_data = processed_movies.loc[train_movieIds, 'genres_decade']
        test_data = processed_movies.loc[test_movieIds, 'genres_decade']
    except Exception as e:
        raise RuntimeError(f"Error extracting data: {e}")
    
    vectorizer = TfidfVectorizer()
    
    try:
        train_tfidf = vectorizer.fit_transform(train_data)
        test_tfidf = vectorizer.transform(test_data)
    except Exception as e:
        raise RuntimeError(f"Error during TF-IDF transformation: {e}")
    
    return vectorizer, train_tfidf, test_tfidf


def tfidf_svd_features(processed_movies, train_movieIds, test_movieIds, n_components=500, variance_ratio=None, random_state=42):
    """
    Create features using TF-IDF and subsequent SVD (TruncatedSVD) for dimensionality reduction.
    
    Parameters
    ----------
    processed_movies : pandas.DataFrame
        DataFrame with movie information, must contain column 'genres_decade_tags' (string: genres + decade + tags)
    train_movieIds : list or array-like
        List of movie IDs for the training set
    test_movieIds : list or array-like
        List of movie IDs for the test set
    n_components : int, default=500
        Number of SVD components
    variance_ratio : float, optional, default=None
        If specified, overrides n_components as the minimum number of components
        explaining the given variance ratio (e.g., 0.95)
    random_state : int, default=42
        Seed for reproducibility
    
    Returns
    -------
    tuple
        (vectorizer, svd, train_features, test_features)
        - vectorizer : TfidfVectorizer - trained vectorizer
        - svd : TruncatedSVD - trained SVD model
        - train_features : numpy.ndarray - features of training movies after SVD
        - test_features : numpy.ndarray - features of test movies after SVD
    
    Raises
    ----------
    TypeError
        If input parameters have incorrect type
    ValueError
        If column 'genres_decade_tags' is missing, or data is empty,
        or n_components <= 0, or variance_ratio is not in range (0, 1)
    RuntimeError
        If TF-IDF or SVD features could not be created
    """
    if not isinstance(processed_movies, pd.DataFrame):
        raise TypeError(f"processed_movies must be pandas.DataFrame, got {type(processed_movies)}")
    
    if 'genres_decade_tags' not in processed_movies.columns:
        raise ValueError("Column 'genres_decade_tags' is missing from processed_movies")
    
    if len(train_movieIds) == 0:
        raise ValueError("train_movieIds cannot be empty")
    
    if len(test_movieIds) == 0:
        raise ValueError("test_movieIds cannot be empty")
    
    if not isinstance(n_components, int) or n_components <= 0:
        raise ValueError(f"n_components must be a positive integer, got {n_components}")
    
    if variance_ratio is not None:
        if not isinstance(variance_ratio, (int, float)) or not 0 < variance_ratio < 1:
            raise ValueError(f"variance_ratio must be in range (0, 1), got {variance_ratio}")
    
    try:
        train_data = processed_movies.loc[train_movieIds, 'genres_decade_tags']
        test_data = processed_movies.loc[test_movieIds, 'genres_decade_tags']
    except Exception as e:
        raise RuntimeError(f"Error extracting data: {e}")
    
    vectorizer = TfidfVectorizer()
    
    try:
        train_tfidf = vectorizer.fit_transform(train_data)
        test_tfidf = vectorizer.transform(test_data)
    except Exception as e:
        raise RuntimeError(f"Error during TF-IDF transformation: {e}")
    
    # Determine optimal number of components if variance_ratio is specified
    if variance_ratio is not None:
        try:
            max_components = min(n_components, train_tfidf.shape[1] - 1)
            if max_components <= 0:
                raise ValueError(f"Insufficient features for SVD: {train_tfidf.shape[1]}")
            
            temp_svd = TruncatedSVD(n_components=max_components, random_state=random_state)
            temp_svd.fit(train_tfidf)
            
            cumsum = np.cumsum(temp_svd.explained_variance_ratio_)
            n_optimal = np.argmax(cumsum >= variance_ratio) + 1
            n_components = n_optimal
        except Exception as e:
            raise RuntimeError(f"Error determining optimal number of components: {e}")
    
    try:
        svd = TruncatedSVD(n_components=n_components, random_state=random_state)
        train_features = svd.fit_transform(train_tfidf)
        test_features = svd.transform(test_tfidf)
    except Exception as e:
        raise RuntimeError(f"Error during SVD transformation: {e}")
    
    return vectorizer, svd, train_features, test_features

### 2.3.2. Creating Subsets for Recommendation Models

To ensure fair experiments, all models use a time-based train/test split per user. The number of users in the train and test sets is the same. The ratio of train to test sizes is 4:1 (80% train, 20% test).

The dataset contains explicit feedback in the form of ratings from 0.5 to 5. These have different meanings:
- ratings from 0.5 to 3 can be considered negative;
- ratings between 3 and 4 are a gray area;
- ratings from 4 to 5 can be considered positive.

The content-based model primarily addresses ranking during user cold-start, recommending movies that best match user preferences when user history is limited.

Therefore, we will compare several content-based models on three data subsets:
1. All ratings (positive, neutral, and negative)
2. Neutral and positive
3. Only positive

In all three cases, the train/test split ratios will be maintained.

The user profile for quality evaluation is based on the single most recent highest-rated movie from the training set.

Relevant movies are all movies that the user interacted with in the test set.

In [17]:
pos_ratings = ratings.loc[ratings['rating'] >= 4.0]

In [18]:
pos_train_ratings, pos_test_ratings, pos_train_movieId_to_idx, pos_test_movieId_to_idx = train_test_time_split(pos_ratings, train_size=0.8, model_type='cb')
pos_train_idx_to_movieId = inverse_dict(pos_train_movieId_to_idx)
pos_test_idx_to_movieId = inverse_dict(pos_test_movieId_to_idx)

In [19]:
print_split_cb_info(pos_train_ratings, pos_test_ratings, pos_train_movieId_to_idx, pos_test_movieId_to_idx, title='положительных (4-5) ')

Всего положительных (4-5) рейтингов: 15_933_136
Размер обучающего набора: 12_666_859
Размер тестового набора: 3_266_277
Количество фильмов в обучающем наборе: 45931
Количество фильмов в тестовом наборе: 41332
Количество пересекающихся в обучающем и тестовом наборах фильмов: 32_090
Всего фильмов: 55_173


In [20]:
nonneg_ratings = ratings.loc[ratings['rating'] >= 3.0]
nonneg_train_ratings, nonneg_test_ratings, nonneg_train_movieId_to_idx, nonneg_test_movieId_to_idx = train_test_time_split(nonneg_ratings, train_size=0.8, model_type='cb')
nonneg_train_idx_to_movieId = inverse_dict(nonneg_train_movieId_to_idx)
nonneg_test_idx_to_movieId = inverse_dict(nonneg_test_movieId_to_idx)

In [21]:
print_split_cb_info(nonneg_train_ratings, nonneg_test_ratings, nonneg_train_movieId_to_idx, nonneg_test_movieId_to_idx, title='(3-5) ')

Всего (3-5) рейтингов: 26_282_794
Размер обучающего набора: 20_946_494
Размер тестового набора: 5_336_300
Количество фильмов в обучающем наборе: 61536
Количество фильмов в тестовом наборе: 58146
Количество пересекающихся в обучающем и тестовом наборах фильмов: 46_128
Всего фильмов: 73_554


In [22]:
train_ratings, test_ratings, train_movieId_to_idx, test_movieId_to_idx = train_test_time_split(ratings, train_size=0.8, model_type='cb')
train_idx_to_movieId = inverse_dict(train_movieId_to_idx)
test_idx_to_movieId = inverse_dict(test_movieId_to_idx)

In [23]:
print_split_cb_info(train_ratings, test_ratings, train_movieId_to_idx, test_movieId_to_idx)

Всего рейтингов: 32_000_204
Размер обучающего набора: 25_520_897
Размер тестового набора: 6_479_307
Количество фильмов в обучающем наборе: 71288
Количество фильмов в тестовом наборе: 68559
Количество пересекающихся в обучающем и тестовом наборах фильмов: 55_415
Всего фильмов: 84_432


### 2.3.3. Baseline Evaluation

A random recommender is used as a baseline model for both content-based and collaborative filtering models. It "predicts" k movies from the training set for each user, and ranking quality metrics are calculated. The baseline is evaluated for each of the three data subsets.

All metrics are evaluated with k=10 recommendations.

In [24]:
gt = get_ground_truth(test_ratings)
rel_userIds = test_ratings['userId'].unique()
train_movies = list(train_movieId_to_idx.keys())

pos_gt = get_ground_truth(pos_test_ratings)
pos_rel_userIds = pos_test_ratings['userId'].unique()
pos_train_movies = list(pos_train_movieId_to_idx.keys())

nonneg_gt = get_ground_truth(nonneg_test_ratings)
nonneg_rel_userIds = nonneg_test_ratings['userId'].unique()
nonneg_train_movies = list(nonneg_train_movieId_to_idx.keys())

In [25]:
random_metrics_at_10 = agg_metrics(*metrics_at_k_random(train_movies, gt, k=10))
pos_random_metrics_at_10 = agg_metrics(*metrics_at_k_random(pos_train_movies, pos_gt, k=10))
nonneg_random_metrics_at_10 = agg_metrics(*metrics_at_k_random(nonneg_train_movies, nonneg_gt, k=10))

Random Baseline: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 200703/200703 [11:26<00:00, 292.38it/s]


In [26]:
baseline_metrics = {
    'pos_ratings_random_baseline': pos_random_metrics_at_10,
    'nonneg_ratings_random_baseline': nonneg_random_metrics_at_10,
    'all_ratings_random_baseline': random_metrics_at_10
}

In [27]:
artifacts_dir = PROJECT_ROOT / paths['artifacts_dir']
notebook_artifacts_dir = artifacts_dir / paths['notebooks_artifacts_dir']
baseline_metrics_path = notebook_artifacts_dir / paths['baseline_metrics']
save_json(baseline_metrics, baseline_metrics_path)

In [28]:
baseline_metrics_df = pd.DataFrame(index=baseline_metrics.keys(), data=baseline_metrics.values())
baseline_metrics_df

,precision@10,recall@10,hit_rate@10,ndcg@10
pos_ratings_random_baseline,0.000345,0.000213,0.003443,0.000348
nonneg_ratings_random_baseline,0.000428,0.000164,0.004245,0.000373
all_ratings_random_baseline,0.000461,0.000137,0.004558,0.000350


### 2.3.4. Training and Evaluation of Content-based Recommendation Models

The following movie features are used:
- one-hot genres;
- TF-IDF on genres and decade;
- TF-IDF on genres, decade, and tags + dimensionality reduction with TruncatedSVD preserving 95% data variance;
- TF-IDF on genres, decade, and tags + dimensionality reduction with TruncatedSVD to 100 components.

The similarity between user profile and test movies is calculated using cosine similarity.

In [29]:
train_movieIds = list(train_movieId_to_idx.keys())
pos_train_movieIds = list(pos_train_movieId_to_idx.keys())
nonneg_train_movieIds = list(nonneg_train_movieId_to_idx.keys())

test_movieIds = list(test_movieId_to_idx.keys()) 
pos_test_movieIds = list(pos_test_movieId_to_idx.keys())
nonneg_test_movieIds = list(nonneg_test_movieId_to_idx.keys())

#### 2.3.4.1. One-hot genres features

In [30]:
onehot_vectorizer, train_onehot_features, test_onehot_features = onehot_genres_features(feature_prep, train_movieIds, test_movieIds)
pos_onehot_vectorizer, pos_train_onehot_features, pos_test_onehot_features = onehot_genres_features(feature_prep, pos_train_movieIds, pos_test_movieIds)
nonneg_onehot_vectorizer, nonneg_train_onehot_features, nonneg_test_onehot_features = onehot_genres_features(feature_prep, nonneg_train_movieIds, nonneg_test_movieIds)

In [31]:
user_profiles_by_1 = make_user_profiles(train_ratings, train_onehot_features, train_movieId_to_idx, num_movies=1, sort_rating=True, sort_timestamp=True)
pos_user_profiles_by_1 = make_user_profiles(pos_train_ratings, pos_train_onehot_features, pos_train_movieId_to_idx, num_movies=1, sort_rating=True, sort_timestamp=True)
nonneg_user_profiles_by_1 = make_user_profiles(nonneg_train_ratings, nonneg_train_onehot_features, nonneg_train_movieId_to_idx, num_movies=1, sort_rating=True, sort_timestamp=True)

In [32]:
oh_genres_metrics_at_10 = agg_metrics(*metrics_at_k_CB(user_profiles_by_1, test_onehot_features, gt, test_idx_to_movieId, k=10, batch_size=1000, verbose=True))

Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 201/201 [02:11<00:00,  1.53it/s]


In [33]:
pos_oh_genres_metrics_at_10 = agg_metrics(*metrics_at_k_CB(pos_user_profiles_by_1, pos_test_onehot_features, pos_gt, pos_test_idx_to_movieId, k=10, batch_size=1000, verbose=True))
nonneg_oh_genres_metrics_at_10 = agg_metrics(*metrics_at_k_CB(nonneg_user_profiles_by_1, nonneg_test_onehot_features, nonneg_gt, nonneg_test_idx_to_movieId, k=10, batch_size=1000, verbose=True))

Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 201/201 [02:12<00:00,  1.52it/s]


In [34]:
cb_metrics_at_10 = {
    'all_ratings_onehot_genres': oh_genres_metrics_at_10, 
    'pos_ratings_onehot_genres': pos_oh_genres_metrics_at_10, 
    'nonneg_ratings_onehot_genres': nonneg_oh_genres_metrics_at_10
}

#### 2.3.4.2. TF-IDF genres + decades

In [35]:
tfidf_gd_vectorizer, train_tfidf_gd, test_tfidf_gd = tfidf_genres_dec_features(feature_prep, train_movieIds, test_movieIds)
pos_tfidf_gd_vectorizer, pos_train_tfidf_gd, pos_test_tfidf_gd = tfidf_genres_dec_features(feature_prep, pos_train_movieIds, pos_test_movieIds)
nonneg_tfidf_gd_vectorizer, nonneg_train_tfidf_gd, nonneg_test_tfidf_gd = tfidf_genres_dec_features(feature_prep, nonneg_train_movieIds, nonneg_test_movieIds)

user_profiles_tfidf_gd_1 = make_user_profiles(train_ratings, train_tfidf_gd, train_movieId_to_idx, num_movies=1, sort_rating=True, sort_timestamp=True)
pos_user_profiles_tfidf_gd_1 = make_user_profiles(pos_train_ratings, pos_train_tfidf_gd, pos_train_movieId_to_idx, num_movies=1, sort_rating=True, sort_timestamp=True)
nonneg_user_profiles_tfidf_gd_1 = make_user_profiles(nonneg_train_ratings, nonneg_train_tfidf_gd, nonneg_train_movieId_to_idx, num_movies=1, sort_rating=True, sort_timestamp=True)

tfidf_gd_metrics_at_10 = agg_metrics(*metrics_at_k_CB(user_profiles_tfidf_gd_1, test_tfidf_gd, gt, test_idx_to_movieId, k=10, batch_size=1000, verbose=True))
pos_tfidf_gd_metrics_at_10 = agg_metrics(*metrics_at_k_CB(pos_user_profiles_tfidf_gd_1, pos_test_tfidf_gd, pos_gt, pos_test_idx_to_movieId, k=10, batch_size=1000, verbose=True))
nonneg_tfidf_gd_metrics_at_10 = agg_metrics(*metrics_at_k_CB(nonneg_user_profiles_tfidf_gd_1, nonneg_test_tfidf_gd, nonneg_gt, nonneg_test_idx_to_movieId, k=10, batch_size=1000, verbose=True))

Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 201/201 [03:37<00:00,  1.08s/it]


In [36]:
cb_metrics_at_10['all_ratings_tfidf_genres_decade'] = tfidf_gd_metrics_at_10
cb_metrics_at_10['pos_ratings_tfidf_genres_decade'] = pos_tfidf_gd_metrics_at_10
cb_metrics_at_10['nonneg_ratings_tfidf_genres_decade'] = nonneg_tfidf_gd_metrics_at_10

#### 2.3.4.3. TF-IDF genres + decades + tags + TruncatedSVD Dimension Reduction (95% variance)

In [37]:
tfidf_gdt_vectorizer, svd, train_tfidf_svd, test_tfidf_svd = tfidf_svd_features(feature_prep, train_movieIds, test_movieIds, n_components=500, variance_ratio=0.95, random_state=seed)
pos_tfidf_gdt_vectorizer, pos_svd, pos_train_tfidf_svd, pos_test_tfidf_svd = tfidf_svd_features(feature_prep, pos_train_movieIds, pos_test_movieIds, n_components=500, variance_ratio=0.95, random_state=seed)
nonneg_tfidf_gdt_vectorizer, nonneg_svd, nonneg_train_tfidf_svd, nonneg_test_tfidf_svd = tfidf_svd_features(feature_prep, nonneg_train_movieIds, nonneg_test_movieIds, n_components=500, variance_ratio=0.95, random_state=seed)

In [38]:
svd

,"n_components n_components: int, default=2Desired dimensionality of output data.If algorithm='arpack', must be strictly less than the number of features.If algorithm='randomized', must be less than or equal to the number of features.The default value is useful for visualisation. For LSA, a value of100 is recommended.",np.int64(1)
,"algorithm algorithm: {'arpack', 'randomized'}, default='randomized'SVD solver to use. Either ""arpack"" for the ARPACK wrapper in SciPy(scipy.sparse.linalg.svds), or ""randomized"" for the randomizedalgorithm due to Halko (2009).",'randomized'
,"n_iter n_iter: int, default=5Number of iterations for randomized SVD solver. Not used by ARPACK. Thedefault is larger than the default in:func:`~sklearn.utils.extmath.randomized_svd` to handle sparsematrices that may have large slowly decaying spectrum.",5
,"n_oversamples n_oversamples: int, default=10Number of oversamples for randomized SVD solver. Not used by ARPACK.See :func:`~sklearn.utils.extmath.randomized_svd` for a completedescription... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD solver.Not used by ARPACK. See :func:`~sklearn.utils.extmath.randomized_svd`for more details... versionadded:: 1.1",'auto'
,"random_state random_state: int, RandomState instance or None, default=NoneUsed during randomized svd. Pass an int for reproducible results acrossmultiple function calls.See :term:`Glossary `.",42
,"tol tol: float, default=0.0Tolerance for ARPACK. 0 means machine precision. Ignored by randomizedSVD solver.",0.0


In [39]:
user_profiles_tfidf_svd_1 = make_user_profiles(train_ratings, train_tfidf_svd, train_movieId_to_idx, num_movies=1, sort_rating=True, sort_timestamp=True)
pos_user_profiles_tfidf_svd_1 = make_user_profiles(pos_train_ratings, pos_train_tfidf_svd, pos_train_movieId_to_idx, num_movies=1, sort_rating=True, sort_timestamp=True)
nonneg_user_profiles_tfidf_svd_1 = make_user_profiles(nonneg_train_ratings, nonneg_train_tfidf_svd, nonneg_train_movieId_to_idx, num_movies=1, sort_rating=True, sort_timestamp=True)

In [40]:
tfidf_svd_metrics_at_10 = agg_metrics(*metrics_at_k_CB(user_profiles_tfidf_svd_1, test_tfidf_svd, gt, test_idx_to_movieId, k=10))
pos_tfidf_svd_metrics_at_10 = agg_metrics(*metrics_at_k_CB(pos_user_profiles_tfidf_svd_1, pos_test_tfidf_svd, pos_gt, pos_test_idx_to_movieId, k=10))
nonneg_tfidf_svd_metrics_at_10 = agg_metrics(*metrics_at_k_CB(nonneg_user_profiles_tfidf_svd_1, nonneg_test_tfidf_svd, nonneg_gt, nonneg_test_idx_to_movieId, k=10))

Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 201/201 [01:07<00:00,  2.98it/s]


In [41]:
cb_metrics_at_10['all_ratings_tfidf_genres_decade_tags_svd'] = tfidf_svd_metrics_at_10
cb_metrics_at_10['pos_ratings_tfidf_genres_decade_tags_svd'] = pos_tfidf_svd_metrics_at_10
cb_metrics_at_10['nonneg_ratings_tfidf_genres_decade_tags_svd'] = nonneg_tfidf_svd_metrics_at_10

#### 2.3.4.3. TF-IDF genres + decades + tags + TruncatedSVD Dimension Reduction (100 components)

In [42]:
tfidf_gdt_vectorizer_wvr, svd_wvr, train_tfidf_svd_wvr, test_tfidf_svd_wvr = tfidf_svd_features(feature_prep, train_movieIds, test_movieIds, n_components=100, random_state=seed)
pos_tfidf_gdt_vectorizer_wvr, pos_svd_wvr, pos_train_tfidf_svd_wvr, pos_test_tfidf_svd_wvr = tfidf_svd_features(feature_prep, pos_train_movieIds, pos_test_movieIds, n_components=100, random_state=seed)
nonneg_tfidf_gdt_vectorizer_wvr, nonneg_svd_wvr, nonneg_train_tfidf_svd_wvr, nonneg_test_tfidf_svd_wvr = tfidf_svd_features(feature_prep, nonneg_train_movieIds, nonneg_test_movieIds, n_components=100, random_state=seed)

In [43]:
svd_wvr

,"n_components n_components: int, default=2Desired dimensionality of output data.If algorithm='arpack', must be strictly less than the number of features.If algorithm='randomized', must be less than or equal to the number of features.The default value is useful for visualisation. For LSA, a value of100 is recommended.",100
,"algorithm algorithm: {'arpack', 'randomized'}, default='randomized'SVD solver to use. Either ""arpack"" for the ARPACK wrapper in SciPy(scipy.sparse.linalg.svds), or ""randomized"" for the randomizedalgorithm due to Halko (2009).",'randomized'
,"n_iter n_iter: int, default=5Number of iterations for randomized SVD solver. Not used by ARPACK. Thedefault is larger than the default in:func:`~sklearn.utils.extmath.randomized_svd` to handle sparsematrices that may have large slowly decaying spectrum.",5
,"n_oversamples n_oversamples: int, default=10Number of oversamples for randomized SVD solver. Not used by ARPACK.See :func:`~sklearn.utils.extmath.randomized_svd` for a completedescription... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD solver.Not used by ARPACK. See :func:`~sklearn.utils.extmath.randomized_svd`for more details... versionadded:: 1.1",'auto'
,"random_state random_state: int, RandomState instance or None, default=NoneUsed during randomized svd. Pass an int for reproducible results acrossmultiple function calls.See :term:`Glossary `.",42
,"tol tol: float, default=0.0Tolerance for ARPACK. 0 means machine precision. Ignored by randomizedSVD solver.",0.0


In [44]:
user_profiles_tfidf_svd_wvr_1 = make_user_profiles(train_ratings, train_tfidf_svd_wvr, train_movieId_to_idx, num_movies=1, sort_rating=True, sort_timestamp=True)
pos_user_profiles_tfidf_svd_wvr_1 = make_user_profiles(pos_train_ratings, pos_train_tfidf_svd_wvr, pos_train_movieId_to_idx, num_movies=1, sort_rating=True, sort_timestamp=True)
nonneg_user_profiles_tfidf_svd_wvr_1 = make_user_profiles(nonneg_train_ratings, nonneg_train_tfidf_svd_wvr, nonneg_train_movieId_to_idx, num_movies=1, sort_rating=True, sort_timestamp=True)

In [45]:
tfidf_svd_wvr_metrics_at_10 = agg_metrics(*metrics_at_k_CB(user_profiles_tfidf_svd_wvr_1, test_tfidf_svd_wvr, gt, test_idx_to_movieId, k=10))
pos_tfidf_svd_wvr_metrics_at_10 = agg_metrics(*metrics_at_k_CB(pos_user_profiles_tfidf_svd_wvr_1, pos_test_tfidf_svd_wvr, pos_gt, pos_test_idx_to_movieId, k=10))
nonneg_tfidf_svd_wvr_metrics_at_10 = agg_metrics(*metrics_at_k_CB(nonneg_user_profiles_tfidf_svd_wvr_1, nonneg_test_tfidf_svd_wvr, nonneg_gt, nonneg_test_idx_to_movieId, k=10))

Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 201/201 [01:26<00:00,  2.33it/s]


In [46]:
cb_metrics_at_10['all_ratings_tfidf_genres_decade_tags_svd_without_variance_ratio'] = tfidf_svd_wvr_metrics_at_10
cb_metrics_at_10['pos_ratings_tfidf_genres_decade_tags_svd_without_variance_ratio'] = pos_tfidf_svd_wvr_metrics_at_10
cb_metrics_at_10['nonneg_ratings_tfidf_genres_decade_tags_svd_without_variance_ratio'] = nonneg_tfidf_svd_wvr_metrics_at_10

#### 2.3.4.5. Comparison with Baseline and Selection of the Best Content-based Model

In [47]:
baseline_metrics_df

,precision@10,recall@10,hit_rate@10,ndcg@10
pos_ratings_random_baseline,0.000345,0.000213,0.003443,0.000348
nonneg_ratings_random_baseline,0.000428,0.000164,0.004245,0.000373
all_ratings_random_baseline,0.000461,0.000137,0.004558,0.000350


In [48]:
cb_metrics_at_10_df = pd.DataFrame(index=cb_metrics_at_10.keys(), data=cb_metrics_at_10.values())
cb_metrics_at_10_df

,precision@10,recall@10,hit_rate@10,ndcg@10
all_ratings_onehot_genres,0.004404,0.002112,0.039811,0.003859
pos_ratings_onehot_genres,0.003456,0.003082,0.032174,0.003303
nonneg_ratings_onehot_genres,0.003603,0.002151,0.032750,0.003180
all_ratings_tfidf_genres_decade,0.006329,0.003537,0.055587,0.005382
pos_ratings_tfidf_genres_decade,0.005253,0.004981,0.048050,0.006001
nonneg_ratings_tfidf_genres_decade,0.006107,0.003872,0.054324,0.005897
all_ratings_tfidf_genres_decade_tags_svd,0.001950,0.000782,0.019493,0.001098
pos_ratings_tfidf_genres_decade_tags_svd,0.004385,0.003570,0.041301,0.005723
nonneg_ratings_tfidf_genres_decade_tags_svd,0.008088,0.004336,0.075390,0.007259
all_ratings_tfidf_genres_decade_tags_svd_without_variance_ratio,0.015825,0.009359,0.128048,0.014409


In [49]:
cb_metrics_path = notebook_artifacts_dir / paths['content_based_metrics']
save_json(cb_metrics_at_10, cb_metrics_path)

In [50]:
best_cb_model_config = {
    'name': 'nonneg_ratings_tfidf_genres_decade_tags_svd_without_variance_ratio',
    'svd_params': {
        'n_components': 100,
        'random_state': seed
    }
}
best_cb_model_config_path = notebook_artifacts_dir / paths['best_cb_model_config']
save_json(best_cb_model_config, best_cb_model_config_path)

All content-based models outperform the baselines across all metrics.

Since this model will be used for cold-start scenarios where there is little information about the user, the priority metrics are recall@10 and hit_rate@10. It's more important that the user receives at least one potentially interesting movie rather than the order of recommendations or precision.

Therefore, the model with the best recall@10 is `pos_ratings_tfidf_genres_decade_tags_svd_without_variance_ratio` (0.013113), with the best hit_rate@10 - `all_ratings_tfidf_genres_decade_tags_svd_without_variance_ratio` (0.127794), and the most balanced model is `nonneg_ratings_tfidf_genres_decade_tags_svd_without_variance_ratio` (recall@10=0.010286, hit_rate@10=0.121936).

The final content-based model selected is `nonneg_ratings_tfidf_genres_decade_tags_svd_without_variance_ratio` as the most balanced one.

### 2.3.5. Training and Evaluation of Collaborative Filtering Recommendation Models

Since the priority in movie recommendations for this project is ranking quality rather than predicting exact user ratings, a model designed for predicting implicit feedback scores is used for recommending movies to users with history. Since the original data contains ratings from 0.5 to 5.0, all negative (0.5-3) and neutral (3-4) ratings are filtered out, while positive ratings are converted to ones.

The algorithm used is Alternating Least Squares, which alternately solves optimization problems for user and item embedding matrices.

The experiment involves tuning hyperparameters for the algorithm on positive ratings. Metrics are also calculated for 10 recommendations.

Since a well-trained algorithm should not recommend movies that users dislike, during inference recommendations are checked for the absence of negative and neutral movies and cleaned from them, as negative examples are mixed with previously unseen items and could theoretically be recommended by the algorithm.

In [51]:
pos_cf_train_ratings, pos_cf_test_ratings, pos_cf_movieId_to_idx, pos_cf_userId_to_idx = train_test_time_split(pos_ratings, train_size=0.8, model_type='cf')
pos_cf_idx_to_movieId = inverse_dict(pos_cf_movieId_to_idx)
pos_cf_idx_to_userId = inverse_dict(pos_cf_userId_to_idx)

print_split_cf_info(pos_cf_train_ratings, pos_cf_test_ratings, pos_cf_movieId_to_idx, pos_cf_userId_to_idx, title='позитивных (4-5) ')

Всего позитивных (4-5) рейтингов: 15_920_700
Размер обучающего набора: 12_666_859
Размер тестового набора: 3_253_841
Количество фильмов: 45931
Количество пользователей: 198979


In [52]:
pos_n_users = len(pos_cf_userId_to_idx) 
pos_n_items = len(pos_cf_movieId_to_idx)

pos_train_rows = [pos_cf_userId_to_idx[uid] for uid in pos_cf_train_ratings['userId']]
pos_train_cols = [pos_cf_movieId_to_idx[mid] for mid in pos_cf_train_ratings['movieId']]
pos_train_data = np.ones(len(pos_cf_train_ratings))

pos_train_user_item_matrix = sparse.csr_matrix((pos_train_data, (pos_train_rows, pos_train_cols)), shape=(pos_n_users, pos_n_items))

pos_weighted_matrix = sparse.csr_matrix(bm25_weight(pos_train_user_item_matrix, K1=100, B=0.8))

pos_test_rows = [pos_cf_userId_to_idx[uid] for uid in pos_cf_test_ratings['userId']]
pos_test_cols = [pos_cf_movieId_to_idx[mid] for mid in pos_cf_test_ratings['movieId']]
pos_test_data = np.ones(len(pos_cf_test_ratings))

pos_test_matrix = sparse.csr_matrix((pos_test_data, (pos_test_rows, pos_test_cols)), shape=(pos_n_users, pos_n_items))

In [53]:
hyperparams = {
    'factors': [50, 100],
    'regularization': [0.1, 0.01],
    'iterations': [50, 100],
    'alpha': [1.0, 10.0],
    'random_state': [seed]
}

n_vars = reduce(lambda x, y: x * y, [len(v) for v in hyperparams.values()])
param_combs = [dict(zip(hyperparams.keys(), comb)) for comb in product(*hyperparams.values())]

In [54]:
pos_userIds = list(pos_cf_userId_to_idx.keys())
cf_metrics_at_10 = {}
for i in range(n_vars):
    model = AlternatingLeastSquares(**param_combs[i])
    model.fit(pos_weighted_matrix)
    curr_metrics = agg_metrics(*metrics_at_k_ALS(model, pos_weighted_matrix, pos_test_matrix, pos_userIds, pos_cf_userId_to_idx, pos_cf_idx_to_userId, pos_cf_movieId_to_idx, pos_cf_idx_to_movieId, k=10, filter_already_liked=True, batch_size=1000))
    cf_metrics_at_10[f'pos_{i}'] = {'parameters': param_combs[i], 'metrics': curr_metrics}

C:\something\aie\aie-shishova\.venv\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/50 [00:00<?, ?it/s]

Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [04:02<00:00,  1.22s/it]


  0%|          | 0/50 [00:00<?, ?it/s]

Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [04:01<00:00,  1.22s/it]


...

Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [05:41<00:00,  1.72s/it]


  0%|          | 0/100 [00:00<?, ?it/s]

Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [05:40<00:00,  1.71s/it]


  0%|          | 0/100 [00:00<?, ?it/s]

Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [05:42<00:00,  1.72s/it]


In [55]:
cf_metrics_at_10_df = pd.DataFrame(index=[pos['parameters'] for pos in cf_metrics_at_10.values()], data=[pos['metrics'] for pos in cf_metrics_at_10.values()])
cf_metrics_at_10_df

,precision@10,recall@10,hit_rate@10,ndcg@10
"{'factors': 50, 'regularization': 0.1, 'iterations': 50, 'alpha': 1.0, 'random_state': 42}",0.087581,0.089257,0.496766,0.111634
"{'factors': 50, 'regularization': 0.1, 'iterations': 50, 'alpha': 10.0, 'random_state': 42}",0.077479,0.078784,0.461158,0.096648
"{'factors': 50, 'regularization': 0.1, 'iterations': 100, 'alpha': 1.0, 'random_state': 42}",0.087493,0.089162,0.496123,0.111544
"{'factors': 50, 'regularization': 0.1, 'iterations': 100, 'alpha': 10.0, 'random_state': 42}",0.077412,0.078473,0.460600,0.096518
"{'factors': 50, 'regularization': 0.01, 'iterations': 50, 'alpha': 1.0, 'random_state': 42}",0.087622,0.089342,0.496912,0.111748
"{'factors': 50, 'regularization': 0.01, 'iterations': 50, 'alpha': 10.0, 'random_state': 42}",0.077473,0.078744,0.461460,0.096611
"{'factors': 50, 'regularization': 0.01, 'iterations': 100, 'alpha': 1.0, 'random_state': 42}",0.087511,0.089154,0.496555,0.111613
"{'factors': 50, 'regularization': 0.01, 'iterations': 100, 'alpha': 10.0, 'random_state': 42}",0.077495,0.078633,0.460686,0.096558
"{'factors': 100, 'regularization': 0.1, 'iterations': 50, 'alpha': 1.0, 'random_state': 42}",0.088647,0.090928,0.512090,0.113258
"{'factors': 100, 'regularization': 0.1, 'iterations': 50, 'alpha': 10.0, 'random_state': 42}",0.079114,0.082861,0.477517,0.099465


In [56]:
cf_metrics_path = notebook_artifacts_dir / paths['collaborative_filtering_metrics']
save_json(cf_metrics_at_10, cf_metrics_path)

In [57]:
best_cf_model_config = {
    'name': 'pos_alternating_least_squares',
    'als_params': cf_metrics_at_10['pos_8']['parameters']
}
best_cf_model_config

{'name': 'pos_alternating_least_squares',
 'als_params': {'factors': 100,
  'regularization': 0.1,
  'iterations': 50,
  'alpha': 1.0,
  'random_state': 42}}

In [58]:
best_cf_model_config_path = notebook_artifacts_dir / paths['best_cf_model_config']
save_json(best_cf_model_config, best_cf_model_config_path)

The best configuration: factors=100, regularization=0.1, iterations=50, alpha=1.0. It achieves the following metrics: precision@10=0.0886, recall@10=0.0909, ndcg@10=0.1133; except for ndcg@10, these are the highest. hit_rate@10=0.512 means that more than half of users receive at least one relevant recommendation in the top-10. Interestingly, increasing alpha from 1.0 to 10.0 worsens all metrics. Increasing factors from 50 to 100 improves quality (~1-2%). Increasing iterations from 50 to 100 does not provide significant improvement, allowing the use of 50 iterations.

## 2.4. Results

Two models were selected for the movie recommendation service:

- **Content-based model** for recommending movies to new users:
    - Subset of neutral and positive ratings (3-5);
    - Features: TF-IDF on genres, decades, and tags with dimensionality reduction using TruncatedSVD to 100 components;
    - Test set metrics:
        - precision@10 - 1.49%
        - recall@10 - 1.03%
        - hit_rate@10 - 12.19%
        - ndcg@10 - 1.45%

- **Collaborative filtering model** for recommending movies to users with rating history:
    - Subset of positive binarized ratings ([4-5] -> 1, [0.5-4) -> 0);
    - Algorithm: Alternating Least Squares from the implicit library;
    - Parameters:
        - factors=100;
        - regularization=0.1;
        - iterations=50;
        - alpha=1.0;
    - Test set metrics:
        - precision@10 - 8.86%
        - recall@10 - 9.09%
        - hit_rate@10 - 51.21%
        - ndcg@10 - 11.33%

All models achieved metric values higher than the best random baseline metrics:
- precision@10 - 0.05%
- recall@10 - 0.02%
- hit_rate@10 - 0.46%
- ndcg@10 - 0.04%

## 2.5. Training Models on All Data and Saving Them

Next, the best models are trained on all available data for use during inference. It is expected that they will perform with quality matching the metrics obtained during the experiments.

In [59]:
cb_unique_movies = nonneg_ratings['movieId'].unique()
cb_movieId_to_idx = {int(movieId): i for i, movieId in enumerate(cb_unique_movies)}
cb_idx_to_movieId = inverse_dict(cb_movieId_to_idx)

cb_userId_to_idx = {int(userId): i for i, userId in enumerate(nonneg_ratings['userId'])}
cb_idx_to_userId = inverse_dict(cb_userId_to_idx)

print(f'Number of movies in the non-negative sample: {len(cb_movieId_to_idx)}')
print(f'Number of users in the non-negative sample: {len(cb_userId_to_idx)}')

Количество фильмов в не негативной выборке: 73554
Количество пользователей в не негативной выборке: 200892


In [60]:
cb_data = feature_prep['genres_decade_tags']
cb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('svd', TruncatedSVD(**best_cb_model_config['svd_params']))
])
cb_features = cb_pipeline.fit_transform(cb_data)

In [61]:
unique_users = pos_ratings['userId'].unique()
cf_userId_to_idx = {int(userId): i for i, userId in enumerate(unique_users)}

unique_movies = pos_ratings['movieId'].unique()
cf_movieId_to_idx = {int(movieId): i for i, movieId in enumerate(unique_movies)}

n_users = len(cf_userId_to_idx) 
n_items = len(cf_movieId_to_idx)

print(f'Number of movies in the positive sample: {n_items}')
print(f'Number of users in the positive sample: {n_users}')

rows = [cf_userId_to_idx[uid] for uid in pos_ratings['userId']]
cols = [cf_movieId_to_idx[mid] for mid in pos_ratings['movieId']]
cf_data = np.ones(len(pos_ratings))

user_item_matrix = sparse.csr_matrix((cf_data, (rows, cols)), shape=(n_users, n_items))

weighted_matrix = sparse.csr_matrix(bm25_weight(user_item_matrix, K1=100, B=0.8))
cf_model = AlternatingLeastSquares(**best_cf_model_config['als_params'])
cf_model.fit(weighted_matrix)

Количество фильмов в позитивной выборке: 55174
Количество пользователей в позитивной выборке: 200726


  0%|          | 0/50 [00:00<?, ?it/s]

In [71]:
models_dir = PROJECT_ROOT / paths['artifacts_dir'] / paths['models_dir']
cb_pipeline_path = models_dir / paths['cb_pipeline']
cf_model_path = models_dir / paths['als_model']
data_features_dir = PROJECT_ROOT / paths['data_features_dir']
movie_features_path = data_features_dir / paths['cb_features']
cf_user_item_path = data_processed_dir / paths['cf_user_item_matrix']

save_data(movie_features_path, cb_features)
save_data(cf_user_item_path, weighted_matrix)
save_model(cb_pipeline, cb_pipeline_path)
save_model(cf_model, cf_model_path)

In [99]:
save_json(cf_userId_to_idx, data_processed_dir / paths['cf_userId_to_idx'])
save_json(cf_movieId_to_idx, data_processed_dir / paths['cf_movieId_to_idx'])
save_json(cb_movieId_to_idx, data_processed_dir / paths['cb_movieId_to_idx'])

In [72]:
neg_cf_ratings_groups = ratings.loc[ratings['rating'] < 4.0].groupby('userId')
neg_movieIds = {int(userId): [int(movieId) for movieId in group['movieId'].to_list()]
                for userId, group in neg_cf_ratings_groups}
cf_neg_movieIds_path = data_processed_dir / paths['neg_cf_movieIds']
save_json(neg_movieIds, cf_neg_movieIds_path)

## 2.6. Example of Recommendation Prediction

In [118]:
from mrh.models import recommend_by_movieId, recommend_by_userId
k = 10
example_movieIds = [1, 200]
recommendations_by_movieId = recommend_by_movieId(movieIds=example_movieIds, 
                                                  movie_features=cb_features, 
                                                  movieId_to_idx=cb_movieId_to_idx, 
                                                  idx_to_movieId=inverse_dict(cb_movieId_to_idx), 
                                                  k=k)
for i in range(len(example_movieIds)):
    print(f'\nTop-{k} recommendations for movie {example_movieIds[i]}:')
    for mid, score in zip(*recommendations_by_movieId[i]):
        print(f'Movie ID: {mid:>6}, score: {score:.4f}')


Топ-10 рекомендаций для фильма 1:
ID фильма:   6867, score: 0.9348
ID фильма: 222376, score: 0.9208
ID фильма: 178621, score: 0.9059
ID фильма:   1349, score: 0.8987
ID фильма: 104757, score: 0.8948
ID фильма: 160579, score: 0.8834
ID фильма:   6650, score: 0.8725
ID фильма:   5539, score: 0.8660
ID фильма:   6538, score: 0.8569
ID фильма: 149522, score: 0.8523

Топ-10 рекомендаций для фильма 200:
ID фильма:    581, score: 0.9092
ID фильма:  71170, score: 0.8922
ID фильма:  80639, score: 0.8567
ID фильма:  25923, score: 0.8539
ID фильма:   8383, score: 0.8457
ID фильма: 152471, score: 0.8085
ID фильма: 151277, score: 0.8075
ID фильма: 193541, score: 0.8063
ID фильма:  99822, score: 0.8031
ID фильма: 218077, score: 0.7990


In [119]:
example_userIds = [6, 214]
recommendations_by_userId = recommend_by_userId(userIds=example_userIds, 
                                                 als_model=cf_model, 
                                                 user_item_matrix=weighted_matrix,
                                                 movieId_to_idx=cf_movieId_to_idx, 
                                                 userId_to_idx=cf_userId_to_idx, 
                                                 idx_to_movieId=inverse_dict(cf_movieId_to_idx), 
                                                 idx_to_userId=inverse_dict(cf_userId_to_idx), 
                                                 k=k,
                                                 neg_movieIds=neg_movieIds)

for i in range(len(example_userIds)):
    print(f'\nTop-{k} recommendations for user {example_userIds[i]}:')
    for mid, score in zip(*recommendations_by_userId[i]):
        print(f'Movie ID: {mid:>6}, score: {score:.4f}')


Топ-10 рекомендаций для пользователя 6:
ID фильма:   3578, score: 0.8045
ID фильма:   3623, score: 0.6945
ID фильма:    110, score: 0.6547
ID фильма:   1722, score: 0.6422
ID фильма:   2947, score: 0.6384
ID фильма:   5378, score: 0.6216
ID фильма:   3717, score: 0.5436
ID фильма:     10, score: 0.5334
ID фильма:   5872, score: 0.5318
ID фильма:   4963, score: 0.5235

Топ-10 рекомендаций для пользователя 214:
ID фильма:    778, score: 0.7437
ID фильма:     32, score: 0.7393
ID фильма:    337, score: 0.5777
ID фильма:    555, score: 0.5775
ID фильма:   1220, score: 0.5727
ID фильма:   2542, score: 0.5634
ID фильма:   5669, score: 0.5606
ID фильма:    253, score: 0.5453
ID фильма:     47, score: 0.5437
ID фильма:   2115, score: 0.5437
